In [35]:
import pandas as pd
import h3

In [36]:
def read_trips_data():
    trips = pd.read_parquet('../inputs/blue_bikes_master_trip_data.parquet', engine='pyarrow')
    hubs = pd.read_csv('../inputs/blue_bikes_hub_data.csv')
    accidents = pd.read_csv('../inputs/accident_data.csv')
    return trips, hubs, accidents

In [37]:
trips, hubs, accidents = read_trips_data()

In [39]:
def lat_lon_to_h3(lat, lon, resolution=9):
    return h3.geo_to_h3(lat, lon, resolution)

In [55]:
def get_hexes_between_points(start_lat, start_lon, end_lat, end_lon, resolution=10):
    # Get H3 indexes along the line between the two points
    line_hexes = h3.(start_lat, start_lon, end_lat, end_lon, resolution)
    
    return line_hexes

# Sample coordinates for start and end points
start_latitude, start_longitude = 37.773972, -122.431297
end_latitude, end_longitude = 37.775993, -122.430685

# Find H3 hexagons between the two points
hexes_between_points = get_hexes_between_points(start_latitude, start_longitude, end_latitude, end_longitude)

print(hexes_between_points)

AttributeError: module 'h3' has no attribute 'h3line'

In [40]:
trips['member_casual'] = ['member' if member == 'Customer' else ('casual' if member == 'Subscriber' else member) for member in trips['member_casual']]

In [41]:
trips['started_at'] = pd.to_datetime(trips['started_at'],format='mixed')
trips['ended_at'] = pd.to_datetime(trips['ended_at'],format='mixed')
trips['trip_length_mins'] = [y - x for x, y in zip(trips['started_at'], trips['ended_at'])]

In [42]:
trips['trip_length_mins'] = (trips['trip_length_mins'].dt.seconds // 60) % 60

In [43]:
trips.head()

,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,legacy_start_station_id,legacy_end_station_id,trip_length_mins
0,2023-05-22 16:16:20,2023-05-22 17:06:23,Williams St at Washington St,D32040,NCAAA - Walnut Ave at Crawford St,B32027,42.306539,-71.107669,42.316902,-71.091946,casual,NaN,NaN,50
1,2023-05-24 08:34:32,2023-05-24 08:56:35,Hyde Square - Barbara St at Centre St,E32003,Ames St at Main St,M32037,42.321765,-71.109842,42.362500,-71.088220,member,NaN,NaN,22
2,2023-05-16 08:50:27,2023-05-16 09:13:17,Hyde Square - Barbara St at Centre St,E32003,Ames St at Main St,M32037,42.321765,-71.109842,42.362500,-71.088220,member,NaN,NaN,22
3,2023-05-16 14:29:21,2023-05-16 14:44:33,Hyde Square - Barbara St at Centre St,E32003,St. Alphonsus St at Tremont St,B32063,42.321765,-71.109842,42.333293,-71.101246,member,NaN,NaN,15
4,2023-05-25 07:26:53,2023-05-25 07:52:34,Williams St at Washington St,D32040,Washington St at Fuller St,C32084,42.306539,-71.107669,42.281986,-71.071479,casual,NaN,NaN,25


In [44]:
hubs.head()

,Unnamed: 0,Number,Name,Latitude,Longitude,District,Public,Total docks,Deployment Year
0,0,K32015,1200 Beacon St,42.344149,-71.114674,Brookline,Yes,15,2021.0
1,1,W32006,160 Arsenal,42.364664,-71.175694,Watertown,Yes,11,2021.0
2,2,A32019,175 N Harvard St,42.364475,-71.128408,Boston,Yes,17,2014.0
3,3,S32035,191 Beacon St,42.380323,-71.108786,Somerville,Yes,19,2018.0
4,4,C32094,2 Hummingbird Lane at Olmsted Green,42.288870,-71.095003,Boston,Yes,17,2020.0


In [45]:
# Apply the function to create the new 'H3Index' column
hubs['hex_res_9'] = hubs.apply(lambda row: lat_lon_to_h3(row['Latitude'], row['Longitude']), axis=1)

In [46]:
accidents.head()

,dispatch_ts,mode_type,location_type,street,xstreet1,xstreet2,x_cord,y_cord,lat,long
0,2015-01-01 00:24:27+00,mv,Intersection,NaN,TRAIN ST,WESTGLOW ST,777243.68,2930930.11,42.289749,-71.052516
1,2015-01-01 03:50:33+00,mv,Street,RIVER ST,WALTER ST,WINTHROP ST,758927.71,2918981.60,42.257078,-71.120106
2,2015-01-01 10:14:13+00,ped,Intersection,NaN,LONDON ST,MERIDIAN ST,780725.19,2961410.17,42.373337,-71.039040
3,2015-01-01 18:23:57+00,bike,Intersection,NaN,OLNEY ST,INWOOD ST,772710.48,2936614.62,42.305412,-71.069164
4,2015-01-01 18:42:19+00,ped,Intersection,NaN,WASHINGTON ST,COLUMBUS AVE,764813.61,2940364.63,42.315808,-71.098291


In [47]:
accidents['dispatch_ts'] = pd.to_datetime(accidents['dispatch_ts'],format='mixed')

In [48]:
# Apply the function to create the new 'H3Index' column
accidents['hex_res_9'] = accidents.apply(lambda row: lat_lon_to_h3(row['lat'], row['long']), axis=1)

In [50]:
accidents

,dispatch_ts,mode_type,location_type,street,xstreet1,xstreet2,x_cord,y_cord,lat,long,hex_res_9
0,2015-01-01 00:24:27+00:00,mv,Intersection,NaN,TRAIN ST,WESTGLOW ST,777243.68,2930930.11,42.289749,-71.052516,892a3066ba7ffff
1,2015-01-01 03:50:33+00:00,mv,Street,RIVER ST,WALTER ST,WINTHROP ST,758927.71,2918981.60,42.257078,-71.120106,892a3064bbbffff
2,2015-01-01 10:14:13+00:00,ped,Intersection,NaN,LONDON ST,MERIDIAN ST,780725.19,2961410.17,42.373337,-71.039040,892a30665cfffff
3,2015-01-01 18:23:57+00:00,bike,Intersection,NaN,OLNEY ST,INWOOD ST,772710.48,2936614.62,42.305412,-71.069164,892a3066bdbffff
4,2015-01-01 18:42:19+00:00,ped,Intersection,NaN,WASHINGTON ST,COLUMBUS AVE,764813.61,2940364.63,42.315808,-71.098291,892a3066a4bffff
...,...,...,...,...,...,...,...,...,...,...,...
33837,2023-06-30 14:42:02+00:00,mv,Intersection,NaN,BLUE HILL AVE,WALK HILL ST,766155.88,2926320.95,42.277254,-71.093576,892a3064d43ffff
33838,2023-06-30 16:26:05+00:00,mv,Street,CENTRE ST,HATHAWAY ST,DUNSTER RD,759800.76,2937968.28,42.309089,-71.116776,892a3064173ffff
33839,2023-06-30 17:42:54+00:00,mv,Intersection,NaN,NEPONSET VALLEY PKWY,READVILLE ST,754715.89,2914523.83,42.245024,-71.136034,892a339a59bffff
33840,2023-06-30 20:07:58+00:00,mv,Street,RAMP,LEVERETT CONNECTOR,MAURICE TOBIN BRIDGE,773738.37,2959972.06,42.370890,-71.062714,892a3066013ffff


In [51]:
accidents.to_csv('accidents.csv')
hubs.to_csv('hubs.csv')